In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from linear_combination import *

# Set a nice default style
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

In [ ]:
path = "Spectral_library_clean.xlsx"
df_clean = pd.read_excel(path)

# Keep a version with NaNs as the "truth"
df_truth = df_clean.copy()

# For building linear combinations, we usually set NaN -> 0
df_for_mix = df_clean.fillna(0)
wavelength = df_for_mix['Wavelength']

     Wavelength  Diatom_Ptricornutum  Diatom_Csimplex  \
0           800              0.00000          0.00000   
1           799              0.00000          0.00000   
2           798              0.00000          0.00000   
3           797              0.00000          0.00000   
4           796              0.00000          0.00000   
..          ...                  ...              ...   
446         354              0.15854          0.03724   
447         353              0.16035          0.03678   
448         352              0.15539          0.03662   
449         351              0.15917          0.03660   
450         350              0.15971          0.03701   

     Chlamydomonas_Cpriscuii  Chlamydomonas_Creindhardtii  \
0                    0.00000                      0.00000   
1                    0.00000                      0.00000   
2                    0.00000                      0.00000   
3                    0.00000                      0.00000   
4         

In [42]:
def data_creation_CNN(N_training_, N_validation_, wavelength_, noise_level):
    '''This function outputs a noisy absorption spectrum, normalized with mean and standard deviation.'''
    mixture_spectra_full, weights_spectra = generate_mixture_spectra(df_for_mix, 2)
    print(mixture_spectra_full)
    mixture_spectra = mixture_spectra_full.loc[:, mixture_spectra_full.columns.str.startswith('mix2')]    
    print(mixture_spectra.shape, mixture_spectra_full.shape)
    X = torch.tensor(mixture_spectra.values, dtype = torch.float32) + noise_level * torch.randn(len(wavelength_))

    y = torch.from_numpy(concentration_data)

    # shuffle first
    perm = torch.randperm(X.shape[0])
    X, y = X[perm], y[perm]

    # compute stats on training portion only
    mean = X[:2*N_training_].mean(dim=0)
    std  = X[:2*N_training_].std(dim=0) + 1e-8

    # normalize full dataset
    X_norm = (X - mean) / std

    # split up datasets
    X_train_, y_train_ = X_norm[:2*N_training_],  y[:2*N_training_]
    X_val_,   y_val_   = X_norm[2*N_training_:],  y[2*N_training_:]
    return X_train_, y_train_, X_val_, y_val_

In [41]:
data_creation_CNN(100, 100, wavelength, df_for_mix)

   Wavelength  Diatom_Ptricornutum  Diatom_Csimplex  Chlamydomonas_Cpriscuii  \
0         800                  0.0              0.0                      0.0   
1         799                  0.0              0.0                      0.0   
2         798                  0.0              0.0                      0.0   
3         797                  0.0              0.0                      0.0   
4         796                  0.0              0.0                      0.0   

   Chlamydomonas_Creindhardtii  Dinoflagellate_Symbiodiniumsp  \
0                          0.0                      -0.000045   
1                          0.0                       0.004540   
2                          0.0                      -0.000485   
3                          0.0                      -0.000942   
4                          0.0                       0.005560   

   Dinoflagellate_Smicroadriaticum  Dinoflagellate_Dtrenchii  \
0                         -0.00113                   0.00002   


/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - MSc AMEP/Semester 2/Machine Learning/Project/machine_learning/linear_combination.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mixtures_df[mix_col_name] = mix_spectrum
/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - MSc AMEP/Semester 2/Machine Learning/Project/machine_learning/linear_combination.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mixtures_df[mix_col_name] = mix_spectrum
/Users/jochem1411/Library/CloudStorage/OneDrive-UvA/Year 1 - M

ValueError: Unable to coerce to Series, length must be 10: given 451